# Prithvi-EO-2.0 Fine-Tuning for Mineral Prospectivity

**GeoMine AI Project** | Great Dyke, Zimbabwe | Cr/PGM Targeting

This notebook fine-tunes IBM/NASA's Prithvi-EO-2.0-300M-TL foundation model on Sentinel-2 chips to predict mineral deposit locations. The goal is to achieve LOTO (Leave-One-Tile-Out) CV PR-AUC >= 0.45, which is the threshold where the signal meaningfully transfers across tiles (a goal that classical logistic regression failed to reach).

## Setup

**Runtime:** Runtime -> Change runtime type -> **T4 GPU** (free) or **A100** (Pro)

**Training data:** Upload `chips.zip` and `chip_meta.json` to Colab. Generated by `scripts/prepare_prithvi_chips.py`.

**Expected training time:** ~30-45 min on T4, ~10-15 min on A100.

## Model choice

We use `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL`:
- 300M parameters, Vision Transformer MAE architecture
- Pre-trained on 4.2M global HLS (Harmonized Landsat Sentinel) samples
- **TL** = Temporal + Location embeddings, which help with cross-tile generalization
- Input: 6 bands (B02, B03, B04, B8A, B11, B12) at 224x224 pixels

In [ ]:
# ─── Install dependencies ───
!pip install -q torch torchvision rasterio numpy scikit-learn matplotlib
!pip install -q huggingface_hub timm einops
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-EO-2.0.git

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─── Upload and extract chips ───
# Run scripts/prepare_prithvi_chips.py locally first, then upload:
# - chips.zip (contains data/processed/chips/ and data/processed/chip_labels/)
# - data/processed/chip_meta.json

from google.colab import files
import os

# Option 1: Upload via Colab UI
# uploaded = files.upload()

# Option 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Adjust these paths to where you uploaded the files
CHIPS_DIR = '/content/drive/MyDrive/geomine/chips'
LABELS_DIR = '/content/drive/MyDrive/geomine/chip_labels'
META_PATH = '/content/drive/MyDrive/geomine/chip_meta.json'

import json
with open(META_PATH) as f:
    meta = json.load(f)
print(f'Loaded {len(meta)} chip records')

from collections import Counter
tile_counts = Counter(f"{m['tile']}_{m['label']}" for m in meta)
for k in sorted(tile_counts):
    print(f'  {k}: {tile_counts[k]}')

In [ ]:
# ─── Download Prithvi weights ───
from huggingface_hub import hf_hub_download

MODEL_ID = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'

# Get the model checkpoint
checkpoint_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename='Prithvi_EO_V2_300M_TL.pt',
    cache_dir='/content/prithvi_cache'
)
print(f'Downloaded: {checkpoint_path}')

# Load the checkpoint
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f'Checkpoint keys: {list(ckpt.keys())[:5]}')

In [ ]:
# ─── Dataset ───
import numpy as np
import rasterio
from torch.utils.data import Dataset, DataLoader

# HLS normalization stats from Prithvi pre-training
# Our chips are in reflectance (0-1), HLS stats are in 0-10000 scale
PRITHVI_MEANS = np.array([775.2290, 1080.9920, 1228.5855, 2497.2180, 2204.2412, 1610.8815]) / 10000.0
PRITHVI_STDS  = np.array([1281.5260, 1270.4814, 1399.4836, 1368.3446, 1291.6764, 1154.5053]) / 10000.0

class MineralChipDataset(Dataset):
    def __init__(self, meta, chips_dir, labels_dir):
        self.meta = meta
        self.chips_dir = chips_dir
        self.labels_dir = labels_dir
        self.means = torch.tensor(PRITHVI_MEANS, dtype=torch.float32).view(6, 1, 1)
        self.stds = torch.tensor(PRITHVI_STDS, dtype=torch.float32).view(6, 1, 1)
    
    def __len__(self):
        return len(self.meta)
    
    def __getitem__(self, idx):
        rec = self.meta[idx]
        chip_path = f"{self.chips_dir}/{rec['chip_id']}.tif"
        label_path = f"{self.labels_dir}/{rec['chip_id']}.tif"
        
        with rasterio.open(chip_path) as src:
            chip = src.read().astype(np.float32)  # (6, 224, 224)
        with rasterio.open(label_path) as src:
            mask = src.read(1).astype(np.float32)  # (224, 224)
        
        chip = torch.from_numpy(chip)
        # Normalize
        chip = (chip - self.means) / self.stds
        
        mask = torch.from_numpy(mask)
        
        return chip, mask, rec['tile'], rec['label']

# Create datasets per LOTO split
tiles = sorted(set(m['tile'] for m in meta))
print(f'Tiles: {tiles}')

In [ ]:
# ─── Model: Prithvi encoder + simple segmentation head ───
import torch.nn as nn

# For simplicity, we build a custom head on top of the Prithvi encoder.
# The full TerraTorch workflow uses a UNet decoder, but for a first pass
# we use a simpler approach: ViT encoder -> upsample -> 1x1 conv.

from prithvi_mae import PrithviMAE  # from the Prithvi package

class PrithviSegmenter(nn.Module):
    def __init__(self, prithvi_ckpt, num_classes=2):
        super().__init__()
        
        # Initialize Prithvi encoder
        self.encoder = PrithviMAE(
            img_size=224,
            patch_size=16,
            in_chans=6,
            embed_dim=1024,
            depth=24,
            num_heads=16,
        )
        
        # Load pre-trained weights
        self.encoder.load_state_dict(prithvi_ckpt, strict=False)
        
        # Freeze encoder initially (fine-tune only head)
        for p in self.encoder.parameters():
            p.requires_grad = False
        
        # Segmentation head: patch tokens -> upsample -> logits
        self.head = nn.Sequential(
            nn.Conv2d(1024, 256, 1),
            nn.GELU(),
            nn.Upsample(scale_factor=16, mode='bilinear', align_corners=False),
            nn.Conv2d(256, num_classes, 1),
        )
    
    def forward(self, x):
        # Get patch embeddings from Prithvi
        features = self.encoder.forward_encoder(x)[0]  # (B, num_patches+1, embed_dim)
        
        # Remove CLS token, reshape to 2D
        B = x.shape[0]
        num_patches = 14 * 14  # 224/16 = 14
        patch_features = features[:, 1:1+num_patches]  # (B, 196, 1024)
        feat_map = patch_features.transpose(1, 2).reshape(B, 1024, 14, 14)
        
        logits = self.head(feat_map)  # (B, 2, 224, 224)
        return logits

# Note: The actual Prithvi API may differ slightly. Check the GitHub repo for current usage:
# https://github.com/NASA-IMPACT/Prithvi-EO-2.0

In [ ]:
# ─── LOTO CV Training Loop ───
from sklearn.metrics import precision_recall_curve, auc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

results = {}

for held_out_tile in tiles:
    print(f'\n{"="*70}')
    print(f'LOTO Fold: Hold out {held_out_tile}')
    print(f'{"="*70}')
    
    train_meta = [m for m in meta if m['tile'] != held_out_tile]
    test_meta = [m for m in meta if m['tile'] == held_out_tile]
    
    train_ds = MineralChipDataset(train_meta, CHIPS_DIR, LABELS_DIR)
    test_ds = MineralChipDataset(test_meta, CHIPS_DIR, LABELS_DIR)
    
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)
    
    # Initialize model
    model = PrithviSegmenter(ckpt).to(device)
    
    # Only train the head initially (5 epochs), then unfreeze encoder (10 epochs)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3,
        weight_decay=0.01
    )
    
    # Class imbalance weighting: positive chips have ~500 positive pixels out of 50176
    pos_weight = torch.tensor([100.0]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    # Stage 1: Train head only
    model.train()
    for epoch in range(5):
        total_loss = 0
        for chip, mask, _, _ in train_loader:
            chip, mask = chip.to(device), mask.to(device)
            logits = model(chip)[:, 1]  # positive class logits
            loss = criterion(logits, mask)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'  Stage 1 Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}')
    
    # Stage 2: Unfreeze encoder, fine-tune end-to-end with lower LR
    for p in model.encoder.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
    
    for epoch in range(10):
        total_loss = 0
        for chip, mask, _, _ in train_loader:
            chip, mask = chip.to(device), mask.to(device)
            logits = model(chip)[:, 1]
            loss = criterion(logits, mask)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'  Stage 2 Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}')
    
    # Evaluation
    model.eval()
    all_y_true, all_y_prob = [], []
    with torch.no_grad():
        for chip, mask, _, _ in test_loader:
            chip, mask = chip.to(device), mask.to(device)
            logits = model(chip)[:, 1]
            probs = torch.sigmoid(logits)
            
            all_y_true.extend(mask.cpu().numpy().flatten())
            all_y_prob.extend(probs.cpu().numpy().flatten())
    
    all_y_true = np.array(all_y_true)
    all_y_prob = np.array(all_y_prob)
    
    if all_y_true.sum() > 0:
        prec, rec, _ = precision_recall_curve(all_y_true, all_y_prob)
        pr_auc = auc(rec, prec)
        baseline = all_y_true.mean()
        print(f'\n  PR-AUC: {pr_auc:.3f} (baseline: {baseline:.4f}, lift: {pr_auc/baseline:.1f}x)')
        results[held_out_tile] = {'pr_auc': float(pr_auc), 'baseline': float(baseline)}
    else:
        print(f'\n  SKIP: no positive pixels in test set')

# Summary
print(f'\n{"="*70}')
print('LOTO CV SUMMARY')
print(f'{"="*70}')
for tile, r in results.items():
    print(f'  {tile}: PR-AUC={r["pr_auc"]:.3f} (baseline={r["baseline"]:.4f})')

if results:
    mean_auc = np.mean([r['pr_auc'] for r in results.values()])
    print(f'\n  Mean LOTO PR-AUC: {mean_auc:.3f}')
    print(f'  Target for defensibility: >= 0.45')
    print(f'  Verdict: {"PASS" if mean_auc >= 0.45 else "FAIL"}')

## Next Steps

1. **If LOTO PR-AUC >= 0.45**: The model generalizes. Run inference on the full tile rasters to produce probability maps. This is the pitch-deck number.

2. **If LOTO PR-AUC < 0.45**: Either (a) the data truly doesn't support cross-tile prediction, or (b) we need more training data. Next steps: label expansion, possibly switch to Clay model for multi-sensor fusion.

3. **Inference on full tiles**: Use sliding window inference to produce a probability raster for the full study area.